## 任务澄清（更具体）
你想要整理 **LangChain 中“模型调用”的分类**，并用可运行的 Notebook 代码演示每一类模型的**统一调用接口**（如 `invoke / batch / stream`），避免依赖真实线上模型（用 Fake 模型演示）。

下面将模型调用按 LangChain 常见抽象分为：
1. **LLM（文本补全）**：输入字符串 → 输出字符串（或 `Generation`）
2. **ChatModel（对话模型）**：输入 `messages` → 输出 `AIMessage`
3. **Embeddings（向量模型）**：输入文本 → 输出向量（list[float]）
4. （补充）**Runnable（可组合调用单元）**：上面模型都属于 Runnable，可链式组合并统一用 `invoke/batch/stream`

=====================================================================
# 模型调用分类

角度1： 按照模型功能的不同
非对话模型：(LLMs / Text Model) from langchain_openai import OpenAI
对话模型： (Chat Models) 推荐使用  from langchain_openai import ChatOpenAI
嵌入模型/向量模型： (Embeddings)   RAG的时候使用


角度2： 按照模型调用时，参数的书写  (api_kay、base_url.....)
1. 硬编码模式： 写死代码中
2. 环境变量模式
3. .env文件模式 推荐使用

# 1.对话模型 ChatOpenAI

In [2]:
import os

from langchain_deepseek import ChatDeepSeek
from langchain_openai import ChatOpenAI, OpenAI
from env_utils import (
    API_KEY,
    BASE_URL,
    MODEL_NAME,
    TEMPERATURE  # 导入 float 类型的温度值
)

# 也可以赋值给环境变量使用
os.environ["OPENAI_API_KEY"] = API_KEY
os.environ["OPENAI_BASE_URL"] = BASE_URL
# 任何大语言模型都匹配 openai
llm = ChatOpenAI(
    model_name=MODEL_NAME,
    temperature=TEMPERATURE,
    # api_key=API_KEY,
    # base_url=BASE_URL
)

# langchain-deepseek deepseek自己的专用
# llm = ChatDeepSeek(
#     model_name=MODEL_NAME,
#     temperature=TEMPERATURE,
#     api_key=API_KEY,
#     api_base=BASE_URL
# )

# 普通调用
# result = llm.invoke('今天天气怎么样')
# print(type(result)) # langchain_core.messages.ai.AIMessage
# print(result)

# 流式调用
result = []
for chunk in llm.stream('今天天气怎么样'):
    print(type(chunk))  # <class 'langchain_core.messages.ai.AIMessageChunk'>
    if chunk.content:
        result.append(chunk.content)
    print(chunk)

print(''.join(result))  # 输出中文



<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019b30c5-ce80-7da2-88f8-458c69328886'
<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019b30c5-ce80-7da2-88f8-458c69328886'
<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019b30c5-ce80-7da2-88f8-458c69328886'
<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019b30c5-ce80-7da2-88f8-458c69328886'
<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_metadata={'model_provider': 'openai'} id='lc_run--019b30c5-ce80-7da2-88f8-458c69328886'
<class 'langchain_core.messages.ai.AIMessageChunk'>
content='' additional_kwargs={} response_me

# 2.对话模型

In [ ]:
!pip install langchain langchain-community

In [ ]:
from langchain_community.llms import FakeListLLM
from langchain_community.chat_models import FakeListChatModel
from langchain_community.embeddings import FakeEmbeddings

from langchain_core.messages import HumanMessage

In [ ]:
taxonomy = {
    "LLM（文本补全）": {
        "典型输入": "str",
        "典型输出": "str",
        "调用方法": ["invoke", "batch", "stream"],
        "说明": "更像传统 completion：给提示词返回一段文本",
    },
    "ChatModel（对话模型）": {
        "典型输入": "list[BaseMessage] 或 ChatPromptValue",
        "典型输出": "AIMessage",
        "调用方法": ["invoke", "batch", "stream"],
        "说明": "面向多轮对话：输入是消息序列（Human/AI/System）",
    },
    "Embeddings（向量模型）": {
        "典型输入": "str 或 list[str]",
        "典型输出": "list[float] 或 list[list[float]]",
        "调用方法": ["embed_query", "embed_documents"],
        "说明": "把文本映射到向量空间，用于检索/聚类/相似度等",
    },
    "Runnable（统一可调用抽象）": {
        "典型输入": "Any",
        "典型输出": "Any",
        "调用方法": ["invoke", "batch", "stream", "ainvoke", "abatch", "astream"],
        "说明": "LangChain v0.2+ 的统一接口；LLM/ChatModel/Chain/Prompt/Parser 都可组合",
    },
}
taxonomy

In [ ]:
llm = FakeListLLM(responses=["这是LLM返回的文本结果。"])
llm.invoke("用一句话介绍 LangChain 的模型调用分类")

In [ ]:
list(llm.stream("流式输出示例："))

In [ ]:
chat = FakeListChatModel(responses=["这是ChatModel返回的对话回复。"])
chat.invoke([HumanMessage(content="用一句话介绍 ChatModel 的输入输出")])

In [ ]:
emb = FakeEmbeddings(size=8)
vec_query = emb.embed_query("LangChain embeddings 示例")
vec_docs = emb.embed_documents(["第一段文本", "第二段文本"])
vec_query, vec_docs
